# Proyecto Final de Minería de Datos
## Bank Marketing — Notebook de Modelado y Evaluación

**Integrantes:** Mariana Arbeláez, María Lucía Cadavid y Juan Manuel Arias

**Fuente:** [UCI Machine Learning Repository — Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

### Alcance de este notebook
Este notebook retoma el conjunto de datos limpio de la Fase 1 (`bank_marketing_limpio_sin_balancear.csv`, generado en `01_Preparacion_Datos_Bank_Marketing.ipynb`) y ejecuta:
1. Exclusión formal de la variable `duration` por motivos de fuga de información (*data leakage*).
2. Partición del dataset en **Train (80%)** y **Test (20%)**, estratificada por la variable objetivo.
3. Análisis de asociación entre variables sobre Train: **Spearman** para las numéricas y **V de Cramér** para las categóricas.
4. Construcción del **pipeline de preparación (`sklearn`)**: escalado numérico y codificación *one-hot* de las categóricas.
5. Balanceo de clases con **SMOTE** (`sampling_strategy=0.5`) aplicado **únicamente sobre el conjunto de entrenamiento**.
6. Entrenamiento y evaluación de **métodos simples** (KNN, Árbol de Decisión, Red Neuronal MLP y SVM lineal) y **métodos de ensamble** (Random Forest, XGBoost y Bagging).
7. Comparativa de métricas en el conjunto de prueba y prueba estadística (ANOVA + Tukey HSD) sobre los resultados de validación cruzada.
8. Exportación con `joblib` del preprocesador, los modelos entrenados y el **pipeline final** (preprocesador + mejor modelo).

In [ ]:
%pip -q install imbalanced-learn scikit-learn pandas numpy scipy statsmodels xgboost seaborn matplotlib

## 1. Importación de Librerías

In [ ]:
from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

# Carpeta donde se guardan los artefactos de este notebook (modelos, pipeline y métricas)
OUTPUT_DIR = Path("entregables_modelado_bank")

## 2. Carga del Dataset Limpio
Cargamos el archivo `bank_marketing_limpio_sin_balancear.csv` generado en `01_Preparacion_Datos_Bank_Marketing.ipynb`. Si el archivo no se encuentra, el notebook se detiene con un error: **no se usa el dataset crudo como sustituto**, porque entrenar sobre datos sin limpiar invalidaría los resultados.

In [ ]:
NOMBRE_ARCHIVO = "bank_marketing_limpio_sin_balancear.csv"

# Ubicaciones posibles del archivo limpio generado en el notebook 01
rutas_candidatas = [
    Path("/content") / NOMBRE_ARCHIVO,                                  # Google Colab (archivo subido)
    Path(NOMBRE_ARCHIVO),                                               # Misma carpeta del notebook
    Path("entregables_preparacion_bank_marketing") / NOMBRE_ARCHIVO,    # Carpeta de salida del notebook 01
]

input_path = next((ruta for ruta in rutas_candidatas if ruta.exists()), None)

if input_path is None:
    raise FileNotFoundError(
        f"No se encontró '{NOMBRE_ARCHIVO}'. Ejecute primero 01_Preparacion_Datos_Bank_Marketing.ipynb "
        "o suba el archivo limpio. Rutas revisadas:\n" + "\n".join(f"  - {ruta}" for ruta in rutas_candidatas)
    )

df = pd.read_csv(input_path)
print(f"Archivo cargado desde: {input_path}")
print("Dimensiones del dataset cargado:", df.shape)
display(df.head())

## 3. Exclusión de la variable `duration` (Fuga de datos / Data Leakage)
Como se estableció en el análisis de negocio, la variable `duration` representa la duración del último contacto telefónico.

Dado que la intención del modelo es predecir si un cliente contratará un depósito **antes** de realizar la llamada, `duration` solo se conoce después de realizar la interacción. Su inclusión artificialmente elevaría la precisión del modelo pero lo inutilizaría en un escenario real de despliegue.

In [ ]:
if "duration" in df.columns:
    df = df.drop(columns=["duration"])
    print("Variable 'duration' eliminada correctamente para prevenir fuga de datos.")
else:
    print("La variable 'duration' ya no se encuentra en el conjunto de datos.")

print(f"Nuevas dimensiones del dataset: {df.shape}")

In [ ]:
# Separar características (X) y variable objetivo (y)
X = df.drop(columns=["y"])
y = df["y"].apply(lambda val: 1 if str(val).strip().lower() in ["yes"] else 0)

print("Dimensiones de X (predictores):", X.shape)
print("Distribución de la variable objetivo (y):\n", y.value_counts(normalize=True) * 100)

## 4. Identificación de tipos de variables para el Pipeline

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)

## 5. Partición del dataset estratificado (80% Entrenamiento, 20% Prueba)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print("\nDistribución en y_train (%):\n", y_train.value_counts(normalize=True) * 100)

### 5.1 Asociación entre variables (solo Train)
Antes de modelar revisamos la asociación entre los predictores y con la variable objetivo. Se calcula **únicamente sobre el conjunto de entrenamiento** para no usar información del conjunto de prueba.

- **Variables numéricas:** coeficiente de **Spearman**, que capta relaciones monótonas y es robusto a valores atípicos.
- **Variables categóricas:** **V de Cramér** (con corrección de sesgo), basada en la prueba chi-cuadrado. Las categorías de variables como `job` o `marital` no tienen un orden natural, por lo que un coeficiente de correlación calculado sobre códigos numéricos no tendría interpretación.

Ambas medidas incluyen la variable objetivo `y`.

In [ ]:
# Conjunto de entrenamiento con la variable objetivo
df_train_eda = X_train.copy()
df_train_eda["y"] = y_train.values

# 1. Correlación de Spearman entre variables numéricas (y la variable objetivo binaria)
spearman_corr = df_train_eda[numeric_features + ["y"]].corr(method="spearman")

# 2. V de Cramér con corrección de sesgo (Bergsma, 2013) para variables categóricas
def cramers_v(x, y):
    tabla = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tabla, correction=False)[0]
    n = tabla.to_numpy().sum()
    r, k = tabla.shape
    phi2_corr = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denominador = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2_corr / denominador) if denominador > 0 else 0.0

variables_cat = categorical_features + ["y"]
cramer_matrix = pd.DataFrame(
    [[cramers_v(df_train_eda[a], df_train_eda[b]) for b in variables_cat] for a in variables_cat],
    index=variables_cat,
    columns=variables_cat
)

# 3. Visualización de ambas matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Variables numéricas — Correlación de Spearman", fontsize=13, fontweight="bold")

sns.heatmap(cramer_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            linewidths=0.5, ax=axes[1])
axes[1].set_title("Variables categóricas — V de Cramér", fontsize=13, fontweight="bold")

plt.suptitle("Asociación entre variables (conjunto de entrenamiento)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# 4. Resumen: asociación de cada predictor con la variable objetivo
asociacion_y = pd.concat([
    spearman_corr["y"].drop("y").abs().rename("Asociación con y").to_frame().assign(Medida="|Spearman|"),
    cramer_matrix["y"].drop("y").rename("Asociación con y").to_frame().assign(Medida="V de Cramér"),
]).sort_values("Asociación con y", ascending=False).round(4)

print("--- Asociación de cada predictor con la variable objetivo ---")
display(asociacion_y)

## 6. Construcción del Pipeline de Preparación (Scikit-Learn)

In [ ]:
# Transformador para numéricas (Escalado)
numeric_transformer = StandardScaler()

# Transformador para categóricas (OneHotEncoder)
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Column Transformer que une ambas transformaciones
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Pipeline de preprocesamiento configurado correctamente.")
preprocessor

## 7. Balanceo Controlado con SMOTE (Train 80%)
Para evitar la pérdida de información de la clase mayoritaria, se descarta el undersampling. Se aplica únicamente un sobremuestreo controlado (**SMOTE**) sobre el conjunto de entrenamiento (80%) fijando `sampling_strategy=0.5`.

Esto incrementa la clase minoritaria (`1`) hasta representar exactamente el 50% del tamaño de la clase mayoritaria (`0`).

In [ ]:
# 1. Ajustar y transformar los datos con el preprocesador
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 2. Aplicar SMOTE prudente (Llevar la clase 1 al 50% de la clase 0)
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_transformed, y_train)

print("Dimensiones de entrenamiento antes del balanceo:", X_train_transformed.shape)
print("Dimensiones de entrenamiento después del SMOTE:", X_train_balanced.shape)
print("\nDistribución final de clases en y_train_balanced:\n", pd.Series(y_train_balanced).value_counts())
print("\nVerificación de proporciones (%):\n", pd.Series(y_train_balanced).value_counts(normalize=True) * 100)

## 8. Métodos Simples de Clasificación

En esta sección entrenamos y evaluamos clasificadores individuales: K-Nearest Neighbors, Árbol de Decisión, Red Neuronal (MLP) y Máquina de Soporte Vectorial (SVM lineal).

In [ ]:
# 1. Inicializar el clasificador KNN
knn_model = KNeighborsClassifier(n_neighbors=5)

# 2. Configurar la estrategia de Validación Cruzada Estratificada
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) en Train ---")
cv_scores_auc = cross_val_score(knn_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc')
cv_scores_acc = cross_val_score(knn_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy')

print(f"ROC AUC promedio: {np.mean(cv_scores_auc):.4f} (+/- {np.std(cv_scores_auc):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc):.4f} (+/- {np.std(cv_scores_acc):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
knn_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred = knn_model.predict(X_test_transformed)
y_pred_proba = knn_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (KNN en Test Independiente) ---")
print(classification_report(y_test, y_pred))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba), 4))

# 6. Visualizar la matriz de confusión final
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - KNN")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Inicializar el clasificador de Árbol de Decisión
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de Árbol de Decisión en Train ---")
cv_scores_auc_dt = cross_val_score(dt_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc')
cv_scores_acc_dt = cross_val_score(dt_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy')

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_dt):.4f} (+/- {np.std(cv_scores_auc_dt):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_dt):.4f} (+/- {np.std(cv_scores_acc_dt):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
dt_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_dt = dt_model.predict(X_test_transformed)
y_pred_proba_dt = dt_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (Árbol de Decisión en Test Independiente) ---")
print(classification_report(y_test, y_pred_dt))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_dt), 4))

# 6. Visualizar la matriz de confusión final
cm_dt = confusion_matrix(y_test, y_pred_dt)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_dt, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - Árbol de Decisión")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Inicializar el clasificador de Red Neuronal (Multi-Layer Perceptron)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de Red Neuronal (MLP) en Train ---")
cv_scores_auc_mlp = cross_val_score(mlp_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
cv_scores_acc_mlp = cross_val_score(mlp_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_mlp):.4f} (+/- {np.std(cv_scores_auc_mlp):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_mlp):.4f} (+/- {np.std(cv_scores_acc_mlp):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
mlp_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_mlp = mlp_model.predict(X_test_transformed)
y_pred_proba_mlp = mlp_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (Red Neuronal en Test Independiente) ---")
print(classification_report(y_test, y_pred_mlp))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_mlp), 4))

# 6. Visualizar la matriz de confusión final
cm_mlp = confusion_matrix(y_test, y_pred_mlp)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_mlp, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - Red Neuronal")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

#### Máquina de Soporte Vectorial (SVM lineal calibrada)
Un SVM con kernel RBF tiene un costo de entrenamiento que crece aproximadamente con el cuadrado del número de muestras (~48.000 filas tras SMOTE), y `probability=True` agrega una validación cruzada interna adicional. Limitar `max_iter` para acortar el tiempo deja el modelo sin converger, lo que hace sus resultados poco confiables.

Por eso se usa un **SVM lineal (`LinearSVC`)**, que escala bien a este tamaño, envuelto en **`CalibratedClassifierCV`** (calibración sigmoide de Platt) para obtener las probabilidades que requieren el ROC AUC y la comparativa con los demás modelos.

In [ ]:
# 1. Inicializar el SVM lineal calibrado para obtener probabilidades
svm_model = CalibratedClassifierCV(
    estimator=LinearSVC(C=1.0, dual=False, max_iter=10000, random_state=42),
    method="sigmoid",
    cv=3
)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de SVM Lineal en Train ---")
cv_scores_auc_svm = cross_val_score(svm_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
cv_scores_acc_svm = cross_val_score(svm_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_svm):.4f} (+/- {np.std(cv_scores_auc_svm):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_svm):.4f} (+/- {np.std(cv_scores_acc_svm):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
svm_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_svm = svm_model.predict(X_test_transformed)
y_pred_proba_svm = svm_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (SVM Lineal en Test Independiente) ---")
print(classification_report(y_test, y_pred_svm))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_svm), 4))

# 6. Visualizar la matriz de confusión final
cm_svm = confusion_matrix(y_test, y_pred_svm)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_svm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - SVM Lineal")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

## 9. Métodos de Ensamble y Aprendizaje Complejo

En esta sección implementamos técnicas de ensamble como Bagging, Random Forest y XGBoost para mejorar el rendimiento del modelado combinando múltiples estimadores.

In [ ]:
# 1. Inicializar el clasificador Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de Random Forest en Train ---")
cv_scores_auc_rf = cross_val_score(rf_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
cv_scores_acc_rf = cross_val_score(rf_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_rf):.4f} (+/- {np.std(cv_scores_auc_rf):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_rf):.4f} (+/- {np.std(cv_scores_acc_rf):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
rf_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_rf = rf_model.predict(X_test_transformed)
y_pred_proba_rf = rf_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (Random Forest en Test Independiente) ---")
print(classification_report(y_test, y_pred_rf))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_rf), 4))

# 6. Visualizar la matriz de confusión final
cm_rf = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - Random Forest")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Inicializar el clasificador XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de XGBoost en Train ---")
cv_scores_auc_xgb = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
cv_scores_acc_xgb = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_xgb):.4f} (+/- {np.std(cv_scores_auc_xgb):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_xgb):.4f} (+/- {np.std(cv_scores_acc_xgb):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
xgb_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_xgb = xgb_model.predict(X_test_transformed)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (XGBoost en Test Independiente) ---")
print(classification_report(y_test, y_pred_xgb))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_xgb), 4))

# 6. Visualizar la matriz de confusión final
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - XGBoost")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

In [ ]:
# 1. Inicializar el clasificador Bagging con Árbol de Decisión como base
bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=10, random_state=42),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

# 2. Configurar la estrategia de Validación Cruzada Estratificada (5 folds)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Evaluar el modelo con Validación Cruzada sobre el set de entrenamiento balanceado
print("--- Evaluación con Validación Cruzada (5-Folds) de Bagging en Train ---")
cv_scores_auc_bag = cross_val_score(bagging_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
cv_scores_acc_bag = cross_val_score(bagging_model, X_train_balanced, y_train_balanced, cv=cv_strategy, scoring='accuracy', n_jobs=-1)

print(f"ROC AUC promedio: {np.mean(cv_scores_auc_bag):.4f} (+/- {np.std(cv_scores_auc_bag):.4f})")
print(f"Exactitud (Accuracy) promedio: {np.mean(cv_scores_acc_bag):.4f} (+/- {np.std(cv_scores_acc_bag):.4f})")

# 4. Ajustar el modelo final en todo el conjunto de entrenamiento balanceado
bagging_model.fit(X_train_balanced, y_train_balanced)

# 5. Evaluar en el conjunto de prueba independiente (Test 20%)
y_pred_bag = bagging_model.predict(X_test_transformed)
y_pred_proba_bag = bagging_model.predict_proba(X_test_transformed)[:, 1]

print("\n--- Reporte de Clasificación Final (Bagging en Test Independiente) ---")
print(classification_report(y_test, y_pred_bag))
print("ROC AUC Final en Test:", round(roc_auc_score(y_test, y_pred_proba_bag), 4))

# 6. Visualizar la matriz de confusión final
cm_bag = confusion_matrix(y_test, y_pred_bag)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_bag, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["No Suscribe (0)", "Suscribe (1)"],
            yticklabels=["No Suscribe (0)", "Suscribe (1)"])
plt.title("Matriz de Confusión Final - Bagging")
plt.ylabel("Clase Real")
plt.xlabel("Clase Predicha")
plt.tight_layout()
plt.show()

## 10. Comparativa de Rendimiento y Evaluación de Métricas

Consolidamos y contrastamos los resultados de todos los modelos entrenados usando múltiples métricas de evaluación sobre el conjunto de prueba independiente.

In [ ]:
# 1. Definir los modelos y sus respectivas predicciones de prueba
modelos = {
    "K-Nearest Neighbors": (y_pred, y_pred_proba),
    "Árbol de Decisión": (y_pred_dt, y_pred_proba_dt),
    "Red Neuronal (MLP)": (y_pred_mlp, y_pred_proba_mlp),
    "SVM Lineal": (y_pred_svm, y_pred_proba_svm),
    "Random Forest": (y_pred_rf, y_pred_proba_rf),
    "XGBoost": (y_pred_xgb, y_pred_proba_xgb),
    "Bagging": (y_pred_bag, y_pred_proba_bag)
}

# 2. Calcular un conjunto ampliado de métricas en el set de prueba para cada modelo
metricas_lista = []

for nombre, (pred, proba) in modelos.items():
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, zero_division=0)
    rec = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)

    metricas_lista.append({
        "Modelo": nombre,
        "Accuracy": round(acc, 4),
        "Precision (Clase 1)": round(prec, 4),
        "Recall (Clase 1)": round(rec, 4),
        "F1-Score (Clase 1)": round(f1, 4),
        "ROC AUC": round(auc, 4)
    })

# Crear DataFrame comparativo
df_comparativa = pd.DataFrame(metricas_lista).sort_values(by="ROC AUC", ascending=False)

print("--- Tabla Comparativa de Métricas de Rendimiento en Test ---")
display(df_comparativa)

# 3. Graficar la comparación de múltiples métricas
df_melted = df_comparativa.melt(id_vars="Modelo", var_name="Métrica", value_name="Valor")

plt.figure(figsize=(12, 7))
sns.barplot(data=df_melted, x="Modelo", y="Valor", hue="Métrica", palette="Set2")
plt.ylim(0.0, 1.05)
plt.title("Comparación Multimétrica de Modelos en el Conjunto de Prueba", fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Puntuación (Rango 0 - 1)")
plt.xlabel("Modelo")
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='lower left', bbox_to_anchor=(0.0, 0.05), ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# 1. Recopilar las métricas de validación cruzada (ROC AUC)

resultados_cv = {
    "KNN": cv_scores_auc,
    "Árbol de Decisión": cv_scores_auc_dt,
    "Red Neuronal (MLP)": cv_scores_auc_mlp,
    "SVM Lineal": cv_scores_auc_svm,
    "Random Forest": cv_scores_auc_rf,
    "XGBoost": cv_scores_auc_xgb,
    "Bagging": cv_scores_auc_bag
}

# 2. Validar los resultados de los modelos

for modelo, scores in resultados_cv.items():
    resultados_cv[modelo] = np.asarray(scores, dtype=float).ravel()

    if len(resultados_cv[modelo]) != 5:
        raise ValueError(f"{modelo} debe tener exactamente 5 resultados de validación cruzada.")

    if not np.isfinite(resultados_cv[modelo]).all():
        raise ValueError(f"{modelo} contiene valores NaN o infinitos.")

# 3. Realizar la prueba ANOVA de una vía

f_stat, p_val = stats.f_oneway(*resultados_cv.values())

print("--- Análisis de Varianza (ANOVA de una vía) ---")
print(f"Estadístico F: {f_stat:.4f}")
print(f"Valor p: {p_val:.4e}")

# 4. Interpretar los resultados del ANOVA

if p_val < 0.05:

    print("Resultado: Existe una diferencia estadísticamente significativa entre las medias de los modelos (p < 0.05).")

    # 5. Preparar los datos para la prueba post-hoc de Tukey

    datos_tukey = [
        {"Modelo": modelo, "ROC_AUC": score}
        for modelo, scores in resultados_cv.items()
        for score in scores
    ]

    df_tukey = pd.DataFrame(datos_tukey)

    # 6. Aplicar la prueba de rangos múltiples de Tukey HSD

    tukey = pairwise_tukeyhsd(
        endog=df_tukey["ROC_AUC"],
        groups=df_tukey["Modelo"],
        alpha=0.05
    )

    print("\n--- Prueba Post-Hoc de Tukey HSD ---")
    print(tukey)

    # 7. Obtener los resultados en formato DataFrame

    resultados_tukey = pd.DataFrame(
        tukey.summary().data[1:],
        columns=tukey.summary().data[0]
    )

    print("\n--- Comparaciones entre modelos ---")
    print(resultados_tukey)

else:
    print("Resultado: No se encontraron diferencias estadísticamente significativas entre las medias de los modelos (p >= 0.05).")

## 11. Exportación de Artefactos

Guardamos con `joblib` en `entregables_modelado_bank/`:
1. El **preprocesador** ajustado sobre Train (escalado + *one-hot*).
2. Cada **modelo entrenado**.
3. El **pipeline final** (`preprocesador → mejor modelo según ROC AUC en Test`), que recibe datos en el formato original del dataset limpio (sin `duration`) y devuelve la predicción. SMOTE no forma parte de este pipeline: el balanceo solo se usa para entrenar, nunca al predecir.
4. La **tabla comparativa** de métricas en Test.

Al final se recarga el pipeline desde disco y se verifica que reproduce el ROC AUC en Test.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Modelos entrenados y el nombre de archivo de cada uno
modelos_entrenados = {
    "K-Nearest Neighbors": (knn_model, "knn"),
    "Árbol de Decisión": (dt_model, "arbol_decision"),
    "Red Neuronal (MLP)": (mlp_model, "red_neuronal_mlp"),
    "SVM Lineal": (svm_model, "svm_lineal"),
    "Random Forest": (rf_model, "random_forest"),
    "XGBoost": (xgb_model, "xgboost"),
    "Bagging": (bagging_model, "bagging")
}

# 2. Guardar el preprocesador y cada modelo
joblib.dump(preprocessor, OUTPUT_DIR / "preprocesador.joblib")
for nombre, (modelo, archivo) in modelos_entrenados.items():
    joblib.dump(modelo, OUTPUT_DIR / f"modelo_{archivo}.joblib")

# 3. Construir y guardar el pipeline final con el mejor modelo según ROC AUC en Test
mejor_modelo_nombre = df_comparativa.iloc[0]["Modelo"]
pipeline_final = Pipeline(steps=[
    ("preprocesamiento", preprocessor),
    ("modelo", modelos_entrenados[mejor_modelo_nombre][0])
])
joblib.dump(pipeline_final, OUTPUT_DIR / "pipeline_final.joblib")

# 4. Guardar la tabla comparativa de métricas
df_comparativa.to_csv(OUTPUT_DIR / "comparativa_metricas_test.csv", index=False)

# 5. Verificación: recargar el pipeline y predecir sobre X_test en su formato original
pipeline_cargado = joblib.load(OUTPUT_DIR / "pipeline_final.joblib")
auc_verificacion = roc_auc_score(y_test, pipeline_cargado.predict_proba(X_test)[:, 1])

print(f"Mejor modelo (ROC AUC en Test): {mejor_modelo_nombre}")
print(f"ROC AUC del pipeline recargado en Test: {auc_verificacion:.4f}")
print(f"\nArchivos guardados en '{OUTPUT_DIR}':")
for ruta in sorted(OUTPUT_DIR.iterdir()):
    print(f"  - {ruta.name}")